In [1]:
import pandas as pd
from torchvision.datasets import MNIST
import torchvision.transforms as tfms
from torch.utils.data import DataLoader 
import torch
import torch.nn as nn 
import torch.optim as optim

### Transform 

In [16]:
tfm = tfms.Compose([
    tfms.ToTensor(),
    tfms.Normalize((0.5,),(0.5,))
]) 

trainset = MNIST(root = "./MNIST",train = True , download = True ,transform = tfm)
testset = MNIST(root = "./MNIST" , train = False , download = True , transform = tfm)

### Preparing dataset for CNN

In [17]:
train_loader = DataLoader(trainset , batch_size = 64 , shuffle = True)
test_loader = DataLoader(testset , batch_size = 64 , shuffle = True)

### Build FNN 

In [18]:
class CNN (nn.Module) :
    def __init__(self ):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            
            # 1st convolutional layer
            nn.Conv2d(1 , 32 , kernel_size = 3 , padding = 1 ),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # 2nd convolutional layer
            nn.Conv2d(32 , 64 , kernel_size = 3 , padding = 1 ),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # 3rd convolutional layer
            nn.Conv2d(64 , 128 , kernel_size = 3 , padding = 1 ),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
            
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(3*3*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
            
        )
    def forward(self,x) :
            x = self.conv_layers(x)
            x = x.view(x.size(0),-1)
            x = self.fc_layer(x)
            return x

In [19]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Trainning 

In [22]:
epochs = 10 
for epoch in range (epochs) :
    model.train()
    epoch_loss = 0.0
    for xb,yb in train_loader :
        optimizer.zero_grad()
        
        otp = model(xb)
        loss = criterion(otp,yb)
        loss.backward()
        optimizer.step()
        
        epoch_loss = epoch_loss + loss.item()

    print(f"epoch{(epoch +1)}/10_avg_loss = {epoch_loss/len(train_loader)}")        

epoch1/10_avg_loss = 0.1537862916339014
epoch2/10_avg_loss = 0.040232610618219034
epoch3/10_avg_loss = 0.029677691876139643
epoch4/10_avg_loss = 0.02370412792685584
epoch5/10_avg_loss = 0.01824645522893314
epoch6/10_avg_loss = 0.01607979349531605
epoch7/10_avg_loss = 0.013065355925978586
epoch8/10_avg_loss = 0.01095335977057065
epoch9/10_avg_loss = 0.010796424074859675
epoch10/10_avg_loss = 0.00736279959285514


### Evaluation 

In [33]:
correct_prediction = 0.0
total_data = 0.0
model.eval()

with torch.no_grad():
    for xb,yb in test_loader :
    
        output = model.forward(xb)
        _ , pred= torch.max(output,1)

        correct_prediction += (pred == yb ).sum().item()
        total_data += yb.size(0)  
print(f"accuracy = {((correct_prediction)/(total_data))*100}% ") 


accuracy = 99.29% 
